<a href="https://colab.research.google.com/github/LCaravaggio/AnalisisCuantitativoAvanzado/blob/main/07_Otros/LAMA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -U lightautoml[all]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.1/475.1 kB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.1/216.1 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.9/587.9 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from lightautoml.automl.presets.tabular_presets import TabularAutoML
from lightautoml.tasks import Task

In [2]:
# Crear un dataset sintético
np.random.seed(42)
N = 1000
X = np.random.randn(N, 10)
y = (X[:,0] + 0.5*X[:,1] + 0.1*np.random.randn(N) > 0).astype(int)

df = pd.DataFrame(X, columns=[f"f{i}" for i in range(10)])
df["TARGET"] = y

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["TARGET"], random_state=42)


In [3]:
task = Task('binary')
roles = {
    'target': 'TARGET',
    'drop': []  # si tienes columnas a eliminar, por ejemplo ID
}


In [4]:
automl = TabularAutoML(
    task=task,
    timeout=300,        # por ejemplo 300 segundos como límite
    cpu_limit=2,
    general_params={"use_algos": [["denselight"]]},   # especificar “denselight”
    nn_params={
        "n_epochs": 10,
        "bs": 64,
        "num_workers": 0,
        "freeze_defaults": True,
        # aquí podrías también pasar hidden_size, drop_rate, etc
    },
    nn_pipeline_params={
        "use_qnt": True,
        "use_te": False
    },
    reader_params={
        'n_jobs': 2,
        'cv': 3,
        'random_state': 42
    }
)

oof_pred = automl.fit_predict(train_df, roles=roles, verbose=1)


[20:39:14] Stdout logging level is INFO.


INFO:lightautoml.automl.presets.base:Stdout logging level is INFO.


[20:39:14] Copying TaskTimer may affect the parent PipelineTimer, so copy will create new unlimited TaskTimer


[20:39:14] Task: binary



INFO:lightautoml.automl.presets.base:Task: binary



...

INFO:lightautoml.automl.base:Layer 1 training completed.



[20:39:36] Automl preset training completed in 21.96 seconds



INFO:lightautoml.automl.presets.base:Automl preset training completed in 21.96 seconds



[20:39:36] Model description:
Final prediction for new objects (level 0) = 
	 1.00000 * (3 averaged models Lvl_0_Pipe_0_Mod_0_TorchNN_denselight_0) 



INFO:lightautoml.automl.presets.base:Model description:
Final prediction for new objects (level 0) = 
	 1.00000 * (3 averaged models Lvl_0_Pipe_0_Mod_0_TorchNN_denselight_0) 



In [5]:
test_pred = automl.predict(test_df)
print("OOF AUC:", roc_auc_score(train_df["TARGET"].values, oof_pred.data[:,0]))
print("Test AUC:", roc_auc_score(test_df["TARGET"].values, test_pred.data[:,0]))

OOF AUC: 0.9937673650221521
Test AUC: 0.994194775297768
